In [ ]:
!pip install -q transformers accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_token'))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto",
)
print("Model loaded.")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded.


text goes in → tokenize → model generates → detokenize → text comes out.

In [ ]:
def query_model(system_prompt, user_prompt, max_new_tokens=1500):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    output = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return response

In [ ]:
test_response = query_model(
    "You are a helpful assistant.",
    "What is 2 + 2? Answer with just the number."
)
print(test_response)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


4


In [ ]:
import os, json, time
import numpy as np

if not os.path.exists("RuleArena"):
    os.system("git clone -q https://github.com/SkyRiver-2000/RuleArena.git")

print("RuleArena cloned.")

RuleArena cloned.


In [ ]:
os.chdir("/content/RuleArena/airline")

exec(open("compute_answer.py").read().split("if __name__")[0])
check_base_tables = load_checking_fee()

print("compute_answer.py loaded.")
print("Fee tables loaded.")

compute_answer.py loaded.
Fee tables loaded.


In [ ]:
def compute_ground_truth(info):
    total_cost, info_dict = compute_answer(
        base_price=info["base_price"],
        direction=info["direction"],
        routine=info["routine"],
        customer_class=info["customer_class"],
        bag_list=[{"id": 0, "name": "ticket", "size": [0,0,0], "weight": 0}] + info["bag_list"],
        check_base_tables=check_base_tables,
    )
    return total_cost, info_dict

In [ ]:
with open("reference_rules.txt") as f:
    REFERENCE_RULES = f.read()

print(f"Rules loaded. Total length: {len(REFERENCE_RULES)} characters")
print(f"First 200 chars: {REFERENCE_RULES[:200]}")

Rules loaded. Total length: 19583 characters
First 200 chars: # Bag fees
All published bag fees apply at each check-in location and are base rates according to travel dates and destination; applicable taxes are not shown.

## Carry-on bags
You're allowed 1 carry


In [ ]:
problems = []
with open("synthesized_problems/comp_0.jsonl") as f:
    for line in f:
        problems.append(json.loads(line))

print(f"Loaded {len(problems)} problems")
print(f"\nFirst problem prompt preview:")
print(problems[0]["prompt"][:300])

Loaded 100 problems

First problem prompt preview:
Sarah is a Main Cabin Class passenger flying from Orlando to Philadelphia with the following items:
1. A backpack: 22 x 13 x 6 inches, 10 lbs;
2. A luggage box: 44 x 22 x 20 inches, 69 lbs;
3. A luggage box: 34 x 18 x 12 inches, 51 lbs;
4. A backpack: 38 x 22 x 16 inches, 84 lbs;
5. A backpack: 38 x


In [ ]:
print(f"Total problems: {len(problems)}")
print(f"\nProblem 0 keys: {problems[0].keys()}")
print(f"\nInfo keys: {problems[0]['info'].keys()}")
print(f"\nBag list example: {problems[0]['info']['bag_list'][0]}")
print(f"\nGround truth computation:")
gt_cost, gt_info = compute_answer(
    base_price=problems[0]["info"]["base_price"],
    direction=problems[0]["info"]["direction"],
    routine=problems[0]["info"]["routine"],
    customer_class=problems[0]["info"]["customer_class"],
    bag_list=[{"id": 0, "name": "ticket", "size": [0,0,0], "weight": 0}] + problems[0]["info"]["bag_list"],
    check_base_tables=check_base_tables,
)
print(f"Ground truth total: ${gt_cost}")
print(f"GT info: {gt_info}")

Total problems: 100

Problem 0 keys: dict_keys(['prompt', 'info'])

Info keys: dict_keys(['base_price', 'customer_class', 'routine', 'direction', 'bag_list'])

Bag list example: {'id': 1, 'name': 'backpack', 'size': [22, 13, 6], 'weight': 10}

Ground truth computation:
Ground truth total: $1445
GT info: {'overweight': [0, 100, 30, 200, 200], 'oversize': [0, 200, 30, 200, 30], 'base': [np.int64(40), np.int64(45), np.int64(150), np.int64(200), np.int64(200)], 'customer_class': 'Main Cabin', 'ticket_price': 180, 'place_of_departure': 'U.S.', 'place_of_arrival': 'U.S.', 'routine': 'U.S.', 'total_cost': np.int64(1445), 'bag_list': [{'id': 1, 'name': 'backpack', 'size': [22, 13, 6], 'weight': 10}, {'id': 2, 'name': 'luggage box', 'size': [44, 22, 20], 'weight': 69}, {'id': 3, 'name': 'luggage box', 'size': [34, 18, 12], 'weight': 51}, {'id': 4, 'name': 'backpack', 'size': [38, 22, 16], 'weight': 84}, {'id': 5, 'name': 'backpack', 'size': [38, 14, 11], 'weight': 90}]}


In [ ]:
print(REFERENCE_RULES[8000:12000])

o, U.S. Virgin Islands, Canada, Mexico, Caribbean (excluding Haiti and Cuba), Central America (excluding Panama)               | $45           | $45         | —               | $0       | $0    |
| Between South America (excluding Guyana, Suriname) and U.S., Puerto Rico, U.S. Virgin Islands, Canada, Mexico, Caribbean (including Haiti, excluding Cuba), Central America | $100          | $100        | $0              | $0       | $0    |
| Between Europe, Israel, Qatar and U.S., Puerto Rico, U.S. Virgin Islands, Canada, Mexico, Caribbean (excluding Cuba), Central America, South America                        | $100          | $100        | $0              | $0       | $0    |
| To or from India, China, Japan, South Korea, Hong Kong, Australia and New Zealand                                                                                           | $100          | $100        | $0              | $0       | $0    |

^Main Plus includes 1 extra free checked bag in addition to the Main Cabin

In [ ]:
print(REFERENCE_RULES[12000:16000])

eraries marketed and operated by American Airlines. If you qualify for complimentary bags based on your AAdvantage® status or oneworld® status, the benefits are based on your highest status level at time of ticketing or check-in.

If your status level is:
* Higher at ticketing than at check-in, show your ticket receipt to the airport agent
* Lower at ticketing than at check-in, current benefits will automatically apply

Free checked bags may not apply to codeshare flights operated by our partners. Visit the website of the airline operating your flight for details.

1st checked bag is complimentary for:
* Eligible AAdvantage® Aviator® and Citi® / AAdvantage® cardmembers (on domestic American Airlines operated itineraries)
* AAdvantage Gold® status
* GOL Diamond Smiles members
* oneworld® Ruby

or when traveling to these destinations (excluding Basic Economy):
* Argentina
* Australia
* Brazil
* Chile
* China
* Colombia
* Cuba
* Ecuador
* El Salvador
* Haiti
* Hong Kong
* India
* Israel
*

In [ ]:
print(REFERENCE_RULES[16000:])

ween Europe, Israel, Qatar and U.S., Puerto Rico, U.S. Virgin Islands, Canada, Mexico, Caribbean, Central America, South America                          | $100          | $100       | $100            | $100       | $100      |
| To or from Australia and New Zealand                                                                                                                         | $100          | $100       | $100            | $100       | $100      |
| To or from India, China, Japan, South Korea and Hong Kong                                                                                                    | $100          | $100       | $100            | $100       | $100      |

##### Over 70 lbs / 32kgs to 100 lbs / 45 kgs

| Regions                                                                                                                                                    | Basic Economy | Main Cabin    | Premium Economy | Business       | First          |
|--------------

In [ ]:
def build_baseline_prompt(problem):
    system = (
        "You are an airline pricing assistant. You will be given the policies of "
        "American Airlines and a passenger's itinerary and baggage details. "
        "Compute the TOTAL COST (flight ticket + all checked bag fees, including "
        "any oversize or overweight charges) according to the policies. "
        "Show your step-by-step reasoning for each bag, then give the final answer "
        "as: FINAL TOTAL: $<amount>"
    )
    user = f"POLICIES:\n{REFERENCE_RULES}\n\nPASSENGER:\n{problem['prompt']}"
    return system, user

Hint injection

In [ ]:
def build_hint_injected_prompt(problem, gt_info=None):
    system = (
        "You are an airline pricing assistant. You will be given the policies of "
        "American Airlines, a passenger's itinerary and baggage details, AND "
        "pre-computed measurements for each bag (total dimensions, oversize status, "
        "weight, and overweight status). Use these pre-computed values together with "
        "the policies to compute the TOTAL COST (flight ticket + all checked bag fees). "
        "Show your step-by-step reasoning for each bag, then give the final answer "
        "as: FINAL TOTAL: $<amount>"
    )
    bag_facts = []
    for bag in problem["info"]["bag_list"]:
        total_dim = sum(bag["size"])
        bag_facts.append(
            f"  - Bag {bag['id']} ({bag['name']}): "
            f"total dimensions = {bag['size'][0]}+{bag['size'][1]}+{bag['size'][2]} = {total_dim} inches "
            f"({'OVERSIZE' if total_dim > 62 else 'not oversize'}); "
            f"weight = {bag['weight']} lbs "
            f"({'OVERWEIGHT (>50 lbs)' if bag['weight'] > 50 else 'not overweight'})"
        )
    precomputed = "PRE-COMPUTED BAG MEASUREMENTS:\n" + "\n".join(bag_facts)
    user = f"POLICIES:\n{REFERENCE_RULES}\n\n{precomputed}\n\nPASSENGER:\n{problem['prompt']}"
    return system, user

In [ ]:
sys_a, usr_a = build_baseline_prompt(problems[0])
print("SYSTEM PROMPT:")
print(sys_a)
print("\nUSER PROMPT (last 500 chars):")
print(usr_a[-500:])

SYSTEM PROMPT:
You are an airline pricing assistant. You will be given the policies of American Airlines and a passenger's itinerary and baggage details. Compute the TOTAL COST (flight ticket + all checked bag fees, including any oversize or overweight charges) according to the policies. Show your step-by-step reasoning for each bag, then give the final answer as: FINAL TOTAL: $<amount>

USER PROMPT (last 500 chars):
 Hong Kong, India, Australia, and New Zealand      | $30                                  | $200                                  |

PASSENGER:
Sarah is a Main Cabin Class passenger flying from Orlando to Philadelphia with the following items:
1. A backpack: 22 x 13 x 6 inches, 10 lbs;
2. A luggage box: 44 x 22 x 20 inches, 69 lbs;
3. A luggage box: 34 x 18 x 12 inches, 51 lbs;
4. A backpack: 38 x 22 x 16 inches, 84 lbs;
5. A backpack: 38 x 14 x 11 inches, 90 lbs;

Sarah's flight ticket is $180.


In [ ]:
sys_b, usr_b = build_hint_injected_prompt(problems[0])
print("USER PROMPT (last 800 chars):")
print(usr_b[-800:])

USER PROMPT (last 800 chars):
otal dimensions = 44+22+20 = 86 inches (OVERSIZE); weight = 69 lbs (OVERWEIGHT (>50 lbs))
  - Bag 3 (luggage box): total dimensions = 34+18+12 = 64 inches (OVERSIZE); weight = 51 lbs (OVERWEIGHT (>50 lbs))
  - Bag 4 (backpack): total dimensions = 38+22+16 = 76 inches (OVERSIZE); weight = 84 lbs (OVERWEIGHT (>50 lbs))
  - Bag 5 (backpack): total dimensions = 38+14+11 = 63 inches (OVERSIZE); weight = 90 lbs (OVERWEIGHT (>50 lbs))

PASSENGER:
Sarah is a Main Cabin Class passenger flying from Orlando to Philadelphia with the following items:
1. A backpack: 22 x 13 x 6 inches, 10 lbs;
2. A luggage box: 44 x 22 x 20 inches, 69 lbs;
3. A luggage box: 34 x 18 x 12 inches, 51 lbs;
4. A backpack: 38 x 22 x 16 inches, 84 lbs;
5. A backpack: 38 x 14 x 11 inches, 90 lbs;

Sarah's flight ticket is $180.


In [ ]:
print("Running problem 0 through both conditions...")
print(f"Ground truth: ${gt_cost}")
print()

response_a = query_model(sys_a, usr_a)
print("=== BASELINE RESPONSE ===")
print(response_a)

Running problem 0 through both conditions...
Ground truth: $1445



/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


=== BASELINE RESPONSE ===
To calculate the total cost, we need to determine the fees for each bag and then add them to the flight ticket cost.

1. The first bag (backpack: 22 x 13 x 6 inches, 10 lbs) is within the size and weight limits for Main Cabin Class. The fee for the first bag is $40.

2. The second bag (luggage box: 44 x 22 x 20 inches, 69 lbs) is also within the size and weight limits for Main Cabin Class. However, since it's the second bag, the fee is $45.

3. The third bag (luggage box: 34 x 18 x 12 inches, 51 lbs) is within the size and weight limits for Main Cabin Class. However, since it's the third bag, the fee is $150.

4. The fourth bag (backpack: 38 x 22 x 16 inches, 84 lbs) is overweight. The fee for an overweight bag is $100 (over 53 lbs / 24 kgs to 70 lbs / 32 kgs).

5. The fifth bag (backpack: 38 x 14 x 11 inches, 90 lbs) is also overweight. The fee for an overweight bag is $100 (over 53 lbs / 24 kgs to 70 lbs / 32 kgs).

Now, let's calculate the total cost:

Flig

In [ ]:
response_b = query_model(sys_b, usr_b)
print("=== HINT-INJECTED RESPONSE ===")
print(response_b)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


=== HINT-INJECTED RESPONSE ===
To calculate the total cost for Sarah's flight, we need to consider the checked bag fees for each bag, as well as the flight ticket.

**Bag 1 (backpack):**
- Total dimensions: 41 inches (not oversize)
- Weight: 10 lbs (not overweight)
- Since the bag is not oversize or overweight, there is no additional fee.

**Bag 2 (luggage box):**
- Total dimensions: 86 inches (OVERSIZE)
- Weight: 69 lbs (OVERWEIGHT (>50 lbs))
- Since the bag is both oversize and overweight, we need to consider the higher fee between oversize and overweight. In this case, the oversize fee is higher, so the additional fee is $200 (oversize).

**Bag 3 (luggage box):**
- Total dimensions: 64 inches (OVERSIZE)
- Weight: 51 lbs (OVERWEIGHT (>50 lbs))
- Since the bag is both oversize and overweight, we need to consider the higher fee between oversize and overweight. In this case, the oversize fee is higher, so the additional fee is $200 (oversize).

**Bag 4 (backpack):**
- Total dimensions: 

In [ ]:
import re

def extract_final_total(response_text):
    # First try: look for explicit FINAL TOTAL marker
    match = re.search(r"FINAL TOTAL:\s*\$?\s*([\d,]+\.?\d*)", response_text, re.IGNORECASE)
    if match:
        return float(match.group(1).replace(",", ""))

    # Fallback: last dollar amount in response
    amounts = re.findall(r"\$\s*([\d,]+\.?\d*)", response_text)
    if amounts:
        return float(amounts[-1].replace(",", ""))

    return None

# Test it on our two responses
pred_a = extract_final_total(response_a)
pred_b = extract_final_total(response_b)

print(f"Ground truth:      ${gt_cost}")
print(f"Baseline pred:     ${pred_a}")
print(f"Hint-injected pred: ${pred_b}")
print(f"Baseline correct:  {abs(pred_a - gt_cost) < 0.5}")
print(f"Hint correct:      {abs(pred_b - gt_cost) < 0.5}")

Ground truth:      $1445
Baseline pred:     $615.0
Hint-injected pred: $1125.0
Baseline correct:  False
Hint correct:      False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/derivation_boundaries', exist_ok=True)
DRIVE_PATH = '/content/drive/MyDrive/derivation_boundaries'
print('Drive mounted. Results will be saved to', DRIVE_PATH)

Mounted at /content/drive
Drive mounted. Results will be saved to /content/drive/MyDrive/derivation_boundaries


In [ ]:
MODEL_NAME = "llama_3.1_8b"
RESULTS_PATH = f"{DRIVE_PATH}/airline_results_{MODEL_NAME}.json"
print("Results will save to:", RESULTS_PATH)

Results will save to: /content/drive/MyDrive/derivation_boundaries/airline_results_llama_3.1_8b.json


In [ ]:
# ── MANUAL RANGE CONTROL ──────────────────────────────────────────
START_IDX = 0   # change this when resuming on a new Colab account
END_IDX = 60
# ──────────────────────────────────────────────────────────────────

def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    elif hasattr(obj, "item"):
        return obj.item()
    else:
        return obj

RESULTS_FILE = f"{DRIVE_PATH}/airline_results_{MODEL_NAME}.json"
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        airline_results = json.load(f)
    done_indices = {r["problem_idx"] for r in airline_results}
    print(f"Resuming: {len(airline_results)} problems already done.")
else:
    airline_results = []
    done_indices = set()
    print("Starting fresh.")

for i, problem in enumerate(problems[START_IDX:END_IDX], start=START_IDX):
    if i in done_indices:
        continue
    print(f"\n--- Airline Problem {i+1}/{END_IDX} ---")
    try:
        gt_cost, gt_info = compute_ground_truth(problem["info"])

        sys_a, usr_a = build_baseline_prompt(problem)
        t0 = time.time()
        response_a = query_model(sys_a, usr_a, max_new_tokens=1500)
        time_a = time.time() - t0
        pred_a = extract_final_total(response_a)

        sys_b, usr_b = build_hint_injected_prompt(problem, gt_info)
        t0 = time.time()
        response_b = query_model(sys_b, usr_b, max_new_tokens=1500)
        time_b = time.time() - t0
        pred_b = extract_final_total(response_b)

        correct_a = (pred_a is not None and abs(pred_a - gt_cost) < 0.5)
        correct_b = (pred_b is not None and abs(pred_b - gt_cost) < 0.5)

        print(f"Ground truth: ${gt_cost}")
        print(f"  A: pred=${pred_a} correct={correct_a} ({time_a:.1f}s)")
        print(f"  B: pred=${pred_b} correct={correct_b} ({time_b:.1f}s)")

        airline_results.append({
            "problem_idx": i, "model": MODEL_NAME, "domain": "airline",
            "prompt": problem["prompt"], "ground_truth": gt_cost,
            "gt_info": gt_info,
            "condition_a_response": response_a, "condition_a_predicted": pred_a, "condition_a_correct": correct_a,
            "condition_b_response": response_b, "condition_b_predicted": pred_b, "condition_b_correct": correct_b,
        })

        with open(RESULTS_FILE, "w") as f:
            json.dump(make_json_safe(airline_results), f, indent=2)
        print(f"  [Saved problem {i+1}]")

    except Exception as e:
        print(f"  ERROR on problem {i}: {e}")
        continue

acc_a = sum(r["condition_a_correct"] for r in airline_results) / len(airline_results)
acc_b = sum(r["condition_b_correct"] for r in airline_results) / len(airline_results)
print(f"\n\n=== AIRLINE SUMMARY ({MODEL_NAME}) ===")
print(f"Condition A accuracy: {acc_a:.1%}")
print(f"Condition B accuracy: {acc_b:.1%}")
print(f"Saved to {RESULTS_FILE}")

os.chdir("/content")

Starting fresh.

--- Airline Problem 1/60 ---
Ground truth: $1445
  A: pred=$615.0 correct=False (59.5s)
  B: pred=$1125.0 correct=False (104.8s)
  [Saved problem 1]

--- Airline Problem 2/60 ---
Ground truth: $1366
  A: pred=$2170.0 correct=False (126.0s)
  B: pred=$316.0 correct=False (96.1s)
  [Saved problem 2]

--- Airline Problem 3/60 ---
Ground truth: $1149
  A: pred=$429.0 correct=False (59.5s)
  B: pred=$1439.0 correct=False (96.8s)
  [Saved problem 3]

--- Airline Problem 4/60 ---
Ground truth: $1713
  A: pred=$1843.0 correct=False (69.7s)
  B: pred=$1683.0 correct=False (165.2s)
  [Saved problem 4]

--- Airline Problem 5/60 ---
Ground truth: $1634
  A: pred=$1434.0 correct=False (102.6s)
  B: pred=$1184.0 correct=False (102.9s)
  [Saved problem 5]

--- Airline Problem 6/60 ---
Ground truth: $2514
  A: pred=$2274.0 correct=False (99.2s)
  B: pred=$2409.0 correct=False (154.8s)
  [Saved problem 6]

--- Airline Problem 7/60 ---
Ground truth: $1331
  A: pred=$1241.0 correct=False